# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook provides an interactive exploration of the FAIR^2 dataset using the [mlcroissant](https://mlcroissant.org) Python library. We'll walk through loading the dataset, inspecting its schema, extracting records, and running exploratory analysis using the Croissant metadata standard.

### Dataset Source
The dataset is described using a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

This dataset represents ordered logistic regression outputs including coefficients, standard errors, p-values, and log likelihood values for predictors of indigenous and modern knowledge adoption in rangeland management practices in Northern Kenya.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and access the dataset package with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (metadata and schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
List the record sets and their associated fields and columns, referencing all entities via their `@id` as specified in the Croissant schema.

Note: RecordSets group related data (like tables); fields map to structured columns with data.

In [ ]:
# Print RecordSet entities and associated fields from the schema

recordsets = dataset.record_sets
if not recordsets:
    print("No record sets detected in this schema. If this is unexpected, check for new releases or updates to the schema.")
else:
    for rs in recordsets:
        print(f"RecordSet: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        elif isinstance(fields, str):
            fields = [fields]
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - {field}")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        elif isinstance(columns, str):
            columns = [columns]
        if columns:
            print("  Columns:")
            for col in columns:
                print(f"    - {col}")
        print("")

## 3. Data Extraction
Load records from each RecordSet into Pandas DataFrames, referencing the RecordSet `@id` and all fields by their Croissant `@id`.

Modify these IDs as needed based on the printed overview above.

In [ ]:
# Determine all available RecordSet IDs
record_sets = [r['@id'] for r in dataset.record_sets]
dataframes = {}

# Attempt to load each RecordSet into a DataFrame by Croissant @id
for record_set_id in record_sets:
    print(f"Loading records from RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set_id])} records.")
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
    print()

# Show columns of the first non-empty DataFrame loaded (if any)
example_df_id = None
for k, v in dataframes.items():
    if not v.empty:
        example_df_id = k
        break

if example_df_id:
    print(f"First record set with data: {example_df_id}")
    print("Columns (by @id):", dataframes[example_df_id].columns.tolist())
    display(dataframes[example_df_id].head())
else:
    print("No dataframes loaded with records.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA using fields (columns) referenced strictly by Croissant `@id`s. Filter records, normalize a numeric field, and optionally group by another field for summary statistics.

*Note: Below is a generic template. Please update `<numeric_field_id>` and `<group_field_id>` using those discovered in section 3 above. Familiar columns: coefficients, log likelihood, p-values, etc.*

In [ ]:
# Update these with actual @id values present in your dataset - see previous code cell's output
numeric_field_id = '<numeric_field_id>' # E.g., '@id' of a coefficient or log likelihood column
group_field_id = '<group_field_id>'     # E.g., '@id' of a categorical column, like region or intervention

# Use example_df_id and DataFrame loaded previously
if example_df_id and numeric_field_id in dataframes[example_df_id].columns:
    threshold = 0  # Adjust as needed for your data

    filtered_df = dataframes[example_df_id][dataframes[example_df_id][numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field (subtract mean, divide by std)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by the group field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id} (grouped):")
        display(grouped_df.head())
    else:
        print(f'Group field {group_field_id} not found in DataFrame.')
else:
    print('No data loaded or specified numeric field id not found.')

## 5. Visualization
Visualize data distributions or relationships using the column `@id`s.

For example, you might plot the distribution of a coefficient or visualize key associations using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_df_id and numeric_field_id in dataframes[example_df_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[example_df_id][numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field found, plot group means (barplot)
    if group_field_id in dataframes[example_df_id].columns:
        plt.figure(figsize=(10, 4))
        group_means = dataframes[example_df_id].groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data or field IDs set for visualization. Please update numeric_field_id and group_field_id.')

## 6. Conclusion
In this notebook, you've learned to:

- Load a Croissant-structured dataset and its metadata using `mlcroissant`
- List and reference all data entities by their unique `@id`
- Extract records and explore them with pandas
- Filter, normalize, and group data in a reproducible way
- Visualize relationships, all through identifiers defined in the Croissant metadata

Explore further: reference the full Croissant schema, try more groupings/fields, or connect the pipeline to your own downstream ML workflows!
